# 🤖 Intent Classification Chatbot using CLINC150
### Built with TF-IDF + Multinomial Naive Bayes + Logistic Regression

---
**Author:** B.Tech Student (NLP & ML Project)  
**Dataset:** CLINC150  
**Goal:** Build an end-to-end chatbot that understands user intent and responds intelligently

---

## 📌 Section 1: Project Introduction

### 🔹 What is NLP (Natural Language Processing)?
NLP is a branch of Artificial Intelligence that helps computers **understand, interpret, and generate human language**.
Examples: Google Translate, Siri, ChatGPT — all use NLP.

### 🔹 What is Intent Classification?
Intent classification means **figuring out what the user wants** from their message.
- User says: *"What is my account balance?"*
- Intent detected: `check_balance`
- The chatbot then gives the appropriate response.

### 🔹 What is a Chatbot?
A chatbot is a software program that **simulates conversation** with users.
- Rule-based chatbots: Follow fixed rules (old style)
- ML-based chatbots: Learn from data and understand context (modern style) ✅ ← We build this!

### 🎯 Project Objectives
1. Load and explore the CLINC150 dataset
2. Preprocess text using NLP techniques
3. Extract features using TF-IDF
4. Train two models: Naive Bayes and Logistic Regression
5. Compare models and select the best one
6. Build a chatbot pipeline with response generation
7. Add fallback handling and conversation logging
8. Create an interactive chat window inside Jupyter

## 📦 Section 2: Import Libraries

In [ ]:
# ============================================================
# INSTALL REQUIRED PACKAGES (run once if not already installed)
# ============================================================
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# Install datasets library to access CLINC150 from HuggingFace
install('datasets')
install('scikit-learn')
install('nltk')
install('seaborn')
install('matplotlib')

print('✅ All packages installed successfully!')

In [ ]:
# ============================================================
# STANDARD LIBRARY IMPORTS
# ============================================================
import os                          # File and directory operations
import re                          # Regular expressions for text cleaning
import csv                         # CSV file writing for conversation logs
import pickle                      # Saving/loading ML models
import random                      # Random response selection
import warnings                    # Suppress unnecessary warnings
from datetime import datetime      # Timestamps for conversation history
from collections import Counter    # Counting intent frequencies

warnings.filterwarnings('ignore')  # Keep output clean

# ============================================================
# DATA HANDLING
# ============================================================
import numpy as np                 # Numerical operations
import pandas as pd                # Data manipulation and analysis

# ============================================================
# DATASET LOADING
# ============================================================
from datasets import load_dataset  # HuggingFace datasets library

# ============================================================
# NLP LIBRARIES
# ============================================================
import nltk                                         # Natural Language Toolkit
from nltk.corpus import stopwords                   # Common words to remove (the, is, a...)
from nltk.tokenize import word_tokenize             # Split text into words
from nltk.stem import PorterStemmer                 # Stemming (running → run)
from nltk.stem import WordNetLemmatizer             # Lemmatization (better than stemming)

# Download necessary NLTK data files
nltk.download('punkt',      quiet=True)  # Tokenizer data
nltk.download('stopwords',  quiet=True)  # Stopwords list
nltk.download('wordnet',    quiet=True)  # WordNet for lemmatization
nltk.download('punkt_tab',  quiet=True)  # Additional tokenizer data
nltk.download('omw-1.4',    quiet=True)  # Open multilingual WordNet

# ============================================================
# MACHINE LEARNING
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF feature extraction
from sklearn.preprocessing import LabelEncoder               # Convert text labels → numbers
from sklearn.model_selection import train_test_split         # Split data for training/testing
from sklearn.naive_bayes import MultinomialNB                # Naive Bayes classifier
from sklearn.linear_model import LogisticRegression          # Logistic Regression classifier
from sklearn.metrics import (                                # Evaluation metrics
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# VISUALIZATION
# ============================================================
import matplotlib.pyplot as plt    # Plotting graphs
import seaborn as sns              # Beautiful statistical charts

# Set consistent plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('✅ All libraries imported successfully!')
print(f'   numpy  : {np.__version__}')
print(f'   pandas : {pd.__version__}')

## 📂 Section 3: Dataset Loading

### 🔹 What is CLINC150?
**CLINC150** is a benchmark dataset for **intent detection** (also called intent classification).

Key facts:
- Created by researchers at **University of California, Berkeley**
- Contains **23,700 queries** across **150 intent classes**
- Covers **10 domains**: banking, travel, home, utility, work, meta, auto, small talk, credit cards, kitchen
- Includes an **"out-of-scope" (oos)** category for unknown queries
- Widely used in NLP research and industry benchmarks

### 🔹 Why CLINC150 for Intent Classification?
✅ Realistic, diverse, real-world queries  
✅ 150 classes = good diversity challenge  
✅ Pre-split into train/validation/test sets  
✅ Publicly available and well-documented

In [ ]:
# ============================================================
# LOAD CLINC150 DATASET
# ============================================================
# We use the HuggingFace 'datasets' library which automatically
# downloads CLINC150 from the internet on first run.
# 'plus' variant includes out-of-scope (oos) queries.

print('⏳ Loading CLINC150 dataset from HuggingFace...')
print('   (First run may take 30-60 seconds to download)\n')

try:
    dataset = load_dataset('clinc_oos', 'plus')
    print('✅ Dataset loaded successfully!\n')
except Exception as e:
    print(f'Error: {e}')
    print('Attempting alternative load method...')
    dataset = load_dataset('clinc_oos', 'small')

# ============================================================
# CONVERT TO PANDAS DATAFRAMES
# ============================================================
# HuggingFace dataset → Pandas DataFrame (easier to work with)

train_df = pd.DataFrame(dataset['train'])       # Training data
valid_df = pd.DataFrame(dataset['validation'])   # Validation data
test_df  = pd.DataFrame(dataset['test'])         # Test data

# The dataset has two columns:
#   'text'  : The user's query (e.g., "What is my balance?")
#   'intent': The intent label as an integer

# Get the list of intent names from the dataset features
intent_names = dataset['train'].features['intent'].names

# Map integer labels → intent name strings
train_df['intent_name'] = train_df['intent'].map(lambda x: intent_names[x])
valid_df['intent_name'] = valid_df['intent'].map(lambda x: intent_names[x])
test_df['intent_name']  = test_df['intent'].map(lambda x: intent_names[x])

print('📊 DATASET OVERVIEW')
print('=' * 50)
print(f'  Training samples   : {len(train_df):,}')
print(f'  Validation samples : {len(valid_df):,}')
print(f'  Test samples       : {len(test_df):,}')
print(f'  Total samples      : {len(train_df) + len(valid_df) + len(test_df):,}')
print(f'  Number of intents  : {len(intent_names)}')
print('=' * 50)

print('\n📋 COLUMN NAMES:', list(train_df.columns))
print('\n🔍 SAMPLE RECORDS (first 5 rows):')
train_df[['text', 'intent_name']].head()

In [ ]:
# ============================================================
# SHOW SAMPLE RECORDS FROM EACH SPLIT
# ============================================================

print('📌 TRAINING DATA SAMPLE:')
print(train_df[['text', 'intent_name']].sample(5, random_state=42).to_string(index=False))

print('\n📌 VALIDATION DATA SAMPLE:')
print(valid_df[['text', 'intent_name']].sample(5, random_state=42).to_string(index=False))

print('\n📌 TEST DATA SAMPLE:')
print(test_df[['text', 'intent_name']].sample(5, random_state=42).to_string(index=False))

print('\n📌 ALL INTENT NAMES:')
print(f'  {intent_names}')

## 📊 Section 4: Exploratory Data Analysis (EDA)

EDA helps us **understand our data** before building models.  
We look at distributions, patterns, and potential issues.

In [ ]:
# ============================================================
# BASIC STATISTICS
# ============================================================

print('📊 DATASET STATISTICS')
print('=' * 50)

# Text length analysis
train_df['text_length'] = train_df['text'].apply(len)
train_df['word_count']  = train_df['text'].apply(lambda x: len(x.split()))

print(f'\n📝 TEXT LENGTH STATS (characters):')
print(train_df['text_length'].describe().round(2).to_string())

print(f'\n📝 WORD COUNT STATS:')
print(train_df['word_count'].describe().round(2).to_string())

print(f'\n🔢 INTENT DISTRIBUTION (top 10):')
intent_counts = train_df['intent_name'].value_counts()
print(intent_counts.head(10).to_string())

print(f'\n✅ Each intent has exactly: {intent_counts.min()}–{intent_counts.max()} training samples')
print(f'   This is a BALANCED dataset — great for training!')

In [ ]:
# ============================================================
# VISUALIZATION 1: Intent Distribution (Top 30)
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(16, 14))

# ---- Plot 1: Top 30 intents by frequency ----
top30 = intent_counts.head(30)
axes[0].barh(top30.index[::-1], top30.values[::-1], color=sns.color_palette('husl', 30))
axes[0].set_xlabel('Number of Training Samples', fontsize=12)
axes[0].set_title('📊 Top 30 Intents by Frequency in Training Data', fontsize=14, fontweight='bold')
axes[0].axvline(x=top30.mean(), color='red', linestyle='--', label=f'Mean = {top30.mean():.0f}')
axes[0].legend(fontsize=11)

# ---- Plot 2: Text length distribution ----
axes[1].hist(train_df['text_length'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(train_df['text_length'].mean(), color='red',    linestyle='--', label=f'Mean = {train_df["text_length"].mean():.1f}')
axes[1].axvline(train_df['text_length'].median(), color='green', linestyle='--', label=f'Median = {train_df["text_length"].median():.1f}')
axes[1].set_xlabel('Text Length (characters)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('📏 Distribution of Query Text Lengths', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)

plt.tight_layout(pad=3.0)
plt.savefig('intent_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved as intent_distribution.png')

In [ ]:
# ============================================================
# VISUALIZATION 2: Dataset Split Comparison
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Pie chart: Dataset split sizes ----
split_sizes = [len(train_df), len(valid_df), len(test_df)]
split_labels = [f'Train\n({len(train_df):,})', f'Validation\n({len(valid_df):,})', f'Test\n({len(test_df):,})']
colors = ['#2ecc71', '#3498db', '#e74c3c']
axes[0].pie(split_sizes, labels=split_labels, colors=colors, autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('📂 Dataset Split Distribution', fontsize=13, fontweight='bold')

# ---- Word count distribution ----
axes[1].hist(train_df['word_count'], bins=20, color='mediumpurple', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Word Count per Query', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('📝 Distribution of Word Counts per Query', fontsize=13, fontweight='bold')
axes[1].axvline(train_df['word_count'].mean(), color='red', linestyle='--',
                label=f'Mean = {train_df["word_count"].mean():.1f} words')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

## 🔧 Section 5: NLP Preprocessing

Raw text is **messy** — it has punctuation, numbers, capital letters, stop words etc.  
We need to **clean and normalize** the text before feeding it to ML models.

### Steps We Apply:
1. **Lowercasing** — `"Hello World"` → `"hello world"`
2. **Remove special characters** — `"hello!?"` → `"hello"`
3. **Remove numbers** — `"call 123"` → `"call"`
4. **Tokenization** — `"hello world"` → `["hello", "world"]`
5. **Stop word removal** — remove `"the", "is", "a"` etc.
6. **Stemming** — `"running"` → `"run"`
7. **Lemmatization** — `"running"` → `"run"` (smarter than stemming)

In [ ]:
# ============================================================
# INITIALIZE NLP TOOLS
# ============================================================

stemmer     = PorterStemmer()                      # For stemming
lemmatizer  = WordNetLemmatizer()                  # For lemmatization
stop_words  = set(stopwords.words('english'))      # English stopwords

# ============================================================
# TEXT PREPROCESSING FUNCTION
# ============================================================

def preprocess_text(text, use_stemming=False, use_lemmatization=True):
    """
    Full NLP preprocessing pipeline for a single text string.
    
    Parameters:
        text             : Raw input string
        use_stemming     : Apply Porter stemming (default: False)
        use_lemmatization: Apply WordNet lemmatization (default: True)
    
    Returns:
        Cleaned, preprocessed string
    """
    
    # ---- STEP 1: Lowercase ----
    # Ensures 'Hello' and 'hello' are treated the same
    text = text.lower()
    
    # ---- STEP 2: Remove URLs ----
    # URLs like http://... add noise
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # ---- STEP 3: Remove special characters and punctuation ----
    # Keep only letters, numbers, spaces
    text = re.sub(r'[^a-z0-9\s]', '', text)
    
    # ---- STEP 4: Remove standalone numbers ----
    # Numbers alone (like '123') don't carry intent information
    text = re.sub(r'\b\d+\b', '', text)
    
    # ---- STEP 5: Remove extra whitespace ----
    text = re.sub(r'\s+', ' ', text).strip()
    
    # ---- STEP 6: Tokenization ----
    # Split text into individual words (tokens)
    tokens = word_tokenize(text)
    
    # ---- STEP 7: Stop word removal ----
    # Remove common words that don't carry intent meaning
    tokens = [word for word in tokens if word not in stop_words and len(word) > 1]
    
    # ---- STEP 8: Stemming OR Lemmatization ----
    # Stemming: fast but crude (running → run)
    # Lemmatization: slower but accurate (better → good)
    if use_stemming:
        tokens = [stemmer.stem(word) for word in tokens]
    elif use_lemmatization:
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # ---- Join tokens back into a string ----
    return ' '.join(tokens)


# ============================================================
# DEMONSTRATE PREPROCESSING WITH EXAMPLES
# ============================================================

demo_texts = [
    "What is my current account BALANCE???",
    "Transfer $500 to John's account immediately!",
    "How are you doing today? I need some help.",
    "My CARD was DECLINED 3 times, what's happening???",
    "Show me the best restaurants near me."
]

print('🔧 TEXT PREPROCESSING DEMONSTRATION')
print('=' * 70)
print(f'{"BEFORE":35} → {"AFTER"}')
print('-' * 70)
for text in demo_texts:
    cleaned = preprocess_text(text)
    print(f'{text[:35]:35} → {cleaned}')
print('=' * 70)

In [ ]:
# ============================================================
# APPLY PREPROCESSING TO ALL DATASETS
# ============================================================

print('⏳ Preprocessing all text data...')

# Apply preprocessing to each split
train_df['clean_text'] = train_df['text'].apply(preprocess_text)
valid_df['clean_text'] = valid_df['text'].apply(preprocess_text)
test_df['clean_text']  = test_df['text'].apply(preprocess_text)

print('✅ Preprocessing complete!\n')

# Show before and after for a few real examples
print('📌 REAL EXAMPLES BEFORE AND AFTER CLEANING:')
print('-' * 70)
sample = train_df[['text', 'clean_text', 'intent_name']].sample(5, random_state=7)
for _, row in sample.iterrows():
    print(f'  Intent      : {row["intent_name"]}')
    print(f'  Original    : {row["text"]}')
    print(f'  Cleaned     : {row["clean_text"]}')
    print()

## 🔢 Section 6: TF-IDF Feature Engineering

### 🔹 What is TF-IDF?
**TF-IDF = Term Frequency × Inverse Document Frequency**

- **Term Frequency (TF)**: How often a word appears in *this* document
- **Inverse Document Frequency (IDF)**: How rare the word is *across all* documents

**Intuition**: A word that appears often in one document but rarely in others is *important* for that document.  
Common words like "the" appear everywhere → low IDF → low TF-IDF score → ignored.  
Unique words like "balance" → high IDF → high TF-IDF score → important!

This converts text into a **numerical matrix** that ML models can understand.

In [ ]:
# ============================================================
# TF-IDF VECTORIZATION
# ============================================================

print('🔢 Creating TF-IDF features...')

tfidf_vectorizer = TfidfVectorizer(
    max_features=15000,    # Keep only top 15,000 most important words
    ngram_range=(1, 2),    # Use single words AND two-word phrases (bigrams)
    min_df=2,              # Ignore words appearing in fewer than 2 documents
    max_df=0.95,           # Ignore words appearing in more than 95% of documents
    sublinear_tf=True      # Apply log scaling to TF (helps with very frequent words)
)

# IMPORTANT: Fit on TRAINING data only!
# Then transform train, validation, and test separately.
# Why? We don't want the model to 'peek' at test data during training.

X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['clean_text'])  # Fit + Transform train
X_valid_tfidf = tfidf_vectorizer.transform(valid_df['clean_text'])        # Only Transform valid
X_test_tfidf  = tfidf_vectorizer.transform(test_df['clean_text'])         # Only Transform test

print(f'\n📊 TF-IDF FEATURE MATRIX:')
print(f'   Training matrix shape   : {X_train_tfidf.shape}  (samples × features)')
print(f'   Validation matrix shape : {X_valid_tfidf.shape}')
print(f'   Test matrix shape       : {X_test_tfidf.shape}')
print(f'   Vocabulary size         : {len(tfidf_vectorizer.vocabulary_):,} unique n-grams')
print(f'   Matrix sparsity         : {100 * (1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])):.1f}%')
print('   (Sparse = most values are 0, which is normal for TF-IDF)')

# ---- Display top features ----
feature_names = tfidf_vectorizer.get_feature_names_out()
print(f'\n🔠 SAMPLE VOCABULARY (first 50 features):')
print(f'   {list(feature_names[:50])}')

In [ ]:
# ============================================================
# TOP TF-IDF WORDS BY AVERAGE SCORE
# ============================================================

# Calculate mean TF-IDF score for each word across all training samples
mean_tfidf = np.asarray(X_train_tfidf.mean(axis=0)).flatten()
top_indices = mean_tfidf.argsort()[-20:][::-1]  # Top 20 highest scoring words

print('🏆 TOP 20 MOST IMPORTANT TF-IDF FEATURES:')
print('-' * 40)
for rank, idx in enumerate(top_indices, 1):
    print(f'  {rank:2}. {feature_names[idx]:25} → score: {mean_tfidf[idx]:.4f}')

## 🏷️ Section 7: Label Encoding

Machine learning models work with **numbers**, not text.  
We need to convert intent labels like `"transfer_money"` → `42` (a number).

**LabelEncoder** from scikit-learn does this automatically.

In [ ]:
# ============================================================
# LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()

# Fit on training labels, then transform all splits
y_train = label_encoder.fit_transform(train_df['intent_name'])  # Fit + encode
y_valid = label_encoder.transform(valid_df['intent_name'])       # Encode using same mapping
y_test  = label_encoder.transform(test_df['intent_name'])        # Encode using same mapping

# How many unique classes?
num_classes = len(label_encoder.classes_)

print(f'✅ Label encoding complete!')
print(f'   Total unique intents (classes): {num_classes}')
print(f'\n📌 LABEL MAPPING EXAMPLES (intent → number):')
print('-' * 50)
# Show first 20 mappings
for idx, class_name in enumerate(label_encoder.classes_[:20]):
    print(f'   {idx:3d} → {class_name}')
print(f'   ... and {num_classes - 20} more intents')

print(f'\n🔍 EXAMPLE ENCODED VALUES:')
print(f'   Training labels (first 10): {y_train[:10]}')
print(f'   Their intent names        : {list(train_df["intent_name"][:10])}')

## ✂️ Section 8: Train-Test Split

### 🔹 Why Do We Need Train and Test Data?

Imagine studying for an exam:
- **Training data** = Practice questions you study from (model learns patterns)
- **Test data** = Actual exam questions (model is evaluated on unseen data)

If we test on the same data we trained on, the model would score 100% — but that's **cheating**!  
We need to see how well it performs on **new, unseen** data.

CLINC150 already provides pre-split data, but we'll also demonstrate manual splitting.

In [ ]:
# ============================================================
# USE THE PRE-SPLIT CLINC150 DATA
# ============================================================
# CLINC150 already comes with train/validation/test splits!
# We'll use these official splits for fairness and reproducibility.

# Our final data variables (already set above):
# X_train_tfidf → TF-IDF features for training
# X_valid_tfidf → TF-IDF features for validation  
# X_test_tfidf  → TF-IDF features for testing
# y_train       → Encoded labels for training
# y_valid       → Encoded labels for validation
# y_test        → Encoded labels for testing

print('📊 FINAL DATA SPLIT SUMMARY')
print('=' * 50)
print(f'  Training   : {X_train_tfidf.shape[0]:,} samples  ← Model learns from this')
print(f'  Validation : {X_valid_tfidf.shape[0]:,} samples  ← Tune hyperparameters')
print(f'  Testing    : {X_test_tfidf.shape[0]:,} samples  ← Final evaluation')
print(f'  Features   : {X_train_tfidf.shape[1]:,} TF-IDF features per sample')
print(f'  Labels     : {num_classes} unique intent classes')
print('=' * 50)
print()
print('🔑 KEY RULE: Never let the model see test data during training!')

## 🔵 Section 9: Train Multinomial Naive Bayes

### 🔹 How Does Naive Bayes Work?
Naive Bayes is based on **Bayes' Theorem** from probability:

`P(Intent | Words) = P(Words | Intent) × P(Intent) / P(Words)`

**In plain English**: Given the words in a sentence, what is the probability of each intent?

**"Naive"** part: It assumes all words are **independent** of each other (not true in reality, but works well in practice!).

**Why Multinomial NB for text?**
- Works well with **word count / frequency data** (like TF-IDF)
- Extremely **fast** to train
- Surprisingly **effective** for text classification
- Great baseline model

In [ ]:
# ============================================================
# TRAIN MULTINOMIAL NAIVE BAYES
# ============================================================

print('🔵 Training Multinomial Naive Bayes...')

nb_model = MultinomialNB(
    alpha=0.1  # Laplace smoothing — prevents zero probabilities for unseen words
               # alpha=1.0 is standard, lower values make the model more aggressive
)

import time
start_time = time.time()

# Train the model!
nb_model.fit(X_train_tfidf, y_train)

train_time_nb = time.time() - start_time

print(f'✅ Naive Bayes training complete in {train_time_nb:.2f} seconds')
print(f'   Classes learned: {nb_model.n_features_in_} TF-IDF features')
print(f'   Number of intents: {len(nb_model.classes_)}')

# ---- Quick accuracy check ----
train_pred_nb = nb_model.predict(X_train_tfidf)
valid_pred_nb = nb_model.predict(X_valid_tfidf)
test_pred_nb  = nb_model.predict(X_test_tfidf)

print(f'\n📊 NAIVE BAYES QUICK RESULTS:')
print(f'   Train Accuracy      : {accuracy_score(y_train, train_pred_nb):.4f} ({accuracy_score(y_train, train_pred_nb)*100:.1f}%)')
print(f'   Validation Accuracy : {accuracy_score(y_valid, valid_pred_nb):.4f} ({accuracy_score(y_valid, valid_pred_nb)*100:.1f}%)')
print(f'   Test Accuracy       : {accuracy_score(y_test, test_pred_nb):.4f}  ({accuracy_score(y_test, test_pred_nb)*100:.1f}%)')

## 🟢 Section 10: Train Logistic Regression

### 🔹 How Does Logistic Regression Work?
Despite the name, Logistic Regression is a **classification** algorithm (not regression!).

It learns a **decision boundary** between classes using the logistic (sigmoid) function:

`P(class) = 1 / (1 + e^(-z))`  where `z = weights × features`

For **multi-class** problems (like ours with 150 classes), it uses **Softmax** to output probabilities for each class.

### 🔹 Why Logistic Regression for NLP?
- Provides **probability scores** (confidence) for each class
- Generally **more accurate** than Naive Bayes for NLP
- Can handle **overlapping** classes better
- Widely used in industry as a strong baseline

In [ ]:
# ============================================================
# TRAIN LOGISTIC REGRESSION
# ============================================================

print('🟢 Training Logistic Regression...')
print('   (This may take 1-3 minutes for 150 classes...)')

lr_model = LogisticRegression(
    max_iter=1000,        # Maximum iterations for convergence
    C=5.0,               # Regularization strength (higher = less regularization)
    solver='saga',        # 'saga' solver works well for large multi-class problems
    multi_class='multinomial',  # Handle 150 classes properly
    n_jobs=-1,           # Use all CPU cores for speed
    random_state=42      # For reproducibility
)

start_time = time.time()
lr_model.fit(X_train_tfidf, y_train)
train_time_lr = time.time() - start_time

print(f'✅ Logistic Regression training complete in {train_time_lr:.2f} seconds')

# ---- Quick accuracy check ----
train_pred_lr = lr_model.predict(X_train_tfidf)
valid_pred_lr = lr_model.predict(X_valid_tfidf)
test_pred_lr  = lr_model.predict(X_test_tfidf)

print(f'\n📊 LOGISTIC REGRESSION QUICK RESULTS:')
print(f'   Train Accuracy      : {accuracy_score(y_train, train_pred_lr):.4f} ({accuracy_score(y_train, train_pred_lr)*100:.1f}%)')
print(f'   Validation Accuracy : {accuracy_score(y_valid, valid_pred_lr):.4f} ({accuracy_score(y_valid, valid_pred_lr)*100:.1f}%)')
print(f'   Test Accuracy       : {accuracy_score(y_test, test_pred_lr):.4f}  ({accuracy_score(y_test, test_pred_lr)*100:.1f}%)')

## 📈 Section 11: Model Evaluation

### 🔹 Evaluation Metrics Explained:
- **Accuracy**: % of correct predictions overall
- **Precision**: Of all predicted as class X, how many actually are X? (Avoids false alarms)
- **Recall**: Of all actual class X, how many did we catch? (Avoids missing real cases)
- **F1 Score**: Harmonic mean of Precision and Recall (balanced metric)

In [ ]:
# ============================================================
# COMPREHENSIVE EVALUATION FUNCTION
# ============================================================

def evaluate_model(model_name, y_true, y_pred):
    """
    Evaluate a model with multiple metrics.
    Returns a dictionary of all metric values.
    """
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    print(f'\n📊 {model_name} — DETAILED EVALUATION ON TEST SET')
    print('=' * 55)
    print(f'  Accuracy  (overall correct) : {acc:.4f}  ({acc*100:.2f}%)')
    print(f'  Precision (weighted avg)    : {prec:.4f}  ({prec*100:.2f}%)')
    print(f'  Recall    (weighted avg)    : {rec:.4f}  ({rec*100:.2f}%)')
    print(f'  F1 Score  (weighted avg)    : {f1:.4f}  ({f1*100:.2f}%)')
    print('=' * 55)
    
    return {'model': model_name, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}


# Evaluate both models on the test set
nb_results = evaluate_model('Multinomial Naive Bayes', y_test, test_pred_nb)
lr_results = evaluate_model('Logistic Regression',    y_test, test_pred_lr)

In [ ]:
# ============================================================
# CLASSIFICATION REPORT (per-intent breakdown)
# ============================================================

print('📋 LOGISTIC REGRESSION — Per-Intent Classification Report (sample):')
print('   (Showing first 20 intents for readability)')
print()

# Get all intent names in the test set
test_intent_labels = label_encoder.inverse_transform(np.unique(y_test))

# Full report
full_report = classification_report(
    y_test, 
    test_pred_lr,
    target_names=label_encoder.classes_,
    zero_division=0
)

# Print first portion (first 20 lines)
report_lines = full_report.split('\n')
print('\n'.join(report_lines[:25]))
print('   ... (truncated for display, full report has all 150 intents)')

In [ ]:
# ============================================================
# CONFUSION MATRIX (subset for readability)
# ============================================================
# With 150 classes, a full confusion matrix would be huge.
# We'll show a confusion matrix for the top 15 most common intents.

print('📊 Generating confusion matrix (top 15 intents)...')

# Get top 15 most frequent intents in test data
top15_intents = pd.Series(label_encoder.inverse_transform(y_test)).value_counts().head(15).index.tolist()
top15_encoded = label_encoder.transform(top15_intents)

# Filter test data to only include these 15 intents
mask = np.isin(y_test, top15_encoded)
y_test_sub   = y_test[mask]
y_pred_sub   = test_pred_lr[mask]

# Compute confusion matrix
cm = confusion_matrix(y_test_sub, y_pred_sub, labels=top15_encoded)

# Plot
fig, ax = plt.subplots(figsize=(14, 10))
short_labels = [name[:20] for name in top15_intents]  # Truncate long names

sns.heatmap(
    cm,
    annot=True, fmt='d',
    xticklabels=short_labels,
    yticklabels=short_labels,
    cmap='Blues',
    linewidths=0.5,
    ax=ax
)

ax.set_title('🔵 Confusion Matrix — Logistic Regression (Top 15 Intents)', fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Predicted Intent', fontsize=12)
ax.set_ylabel('Actual Intent', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved as confusion_matrix.png')

## 🏆 Section 12: Model Comparison

Let's compare both models side-by-side and automatically select the best one.

In [ ]:
# ============================================================
# MODEL COMPARISON TABLE
# ============================================================

comparison_data = {
    'Model'           : [nb_results['model'],   lr_results['model']],
    'Accuracy (%)'    : [f"{nb_results['accuracy']*100:.2f}",  f"{lr_results['accuracy']*100:.2f}"],
    'Precision (%)'   : [f"{nb_results['precision']*100:.2f}", f"{lr_results['precision']*100:.2f}"],
    'Recall (%)'      : [f"{nb_results['recall']*100:.2f}",    f"{lr_results['recall']*100:.2f}"],
    'F1 Score (%)'    : [f"{nb_results['f1']*100:.2f}",        f"{lr_results['f1']*100:.2f}"],
    'Training Time'   : [f"{train_time_nb:.2f}s",              f"{train_time_lr:.2f}s"]
}

comparison_df = pd.DataFrame(comparison_data)
print('\n📊 MODEL COMPARISON TABLE')
print(comparison_df.to_string(index=False))

# ---- Automatically select the best model ----
if lr_results['f1'] >= nb_results['f1']:
    best_model       = lr_model
    best_model_name  = 'Logistic Regression'
    best_f1          = lr_results['f1']
else:
    best_model       = nb_model
    best_model_name  = 'Multinomial Naive Bayes'
    best_f1          = nb_results['f1']

print(f'\n🏆 BEST MODEL: {best_model_name}')
print(f'   F1 Score   : {best_f1*100:.2f}%')
print(f'   This model will be used for the chatbot!')

In [ ]:
# ============================================================
# VISUALIZATION: MODEL COMPARISON BAR CHART
# ============================================================

metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
nb_scores = [nb_results['accuracy'], nb_results['precision'], nb_results['recall'], nb_results['f1']]
lr_scores = [lr_results['accuracy'], lr_results['precision'], lr_results['recall'], lr_results['f1']]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))

bars1 = ax.bar(x - width/2, [s*100 for s in nb_scores], width,
               label='Multinomial Naive Bayes', color='#3498db', alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + width/2, [s*100 for s in lr_scores], width,
               label='Logistic Regression',    color='#2ecc71', alpha=0.85, edgecolor='white')

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#2c3e50')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#2c3e50')

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('🏆 Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 110)
ax.legend(fontsize=11)
ax.axhline(y=90, color='red', linestyle='--', alpha=0.4, label='90% threshold')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Model comparison chart saved as model_comparison.png')

## 💾 Section 13: Save Model

### 🔹 Why Save Models?
Training takes time and compute resources. After training:
- We **save** the model to disk using `pickle`
- Later we can **load** it instantly without retraining
- We can **deploy** it in a web app, API, or production system
- We save the **vectorizer** and **encoder** too — they must match the model!

In [ ]:
# ============================================================
# SAVE THE BEST MODEL AND PREPROCESSING OBJECTS
# ============================================================

# Files to save:
MODEL_FILE     = 'best_model.pkl'          # The trained classifier
VECTORIZER_FILE = 'tfidf_vectorizer.pkl'   # The TF-IDF vectorizer (must use same one!)
ENCODER_FILE   = 'label_encoder.pkl'       # The label encoder (maps numbers ↔ intent names)

# Save with pickle (Python's built-in serialization library)
with open(MODEL_FILE, 'wb') as f:          # 'wb' = write binary mode
    pickle.dump(best_model, f)
print(f'✅ Best model saved       → {MODEL_FILE}')

with open(VECTORIZER_FILE, 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print(f'✅ TF-IDF vectorizer saved → {VECTORIZER_FILE}')

with open(ENCODER_FILE, 'wb') as f:
    pickle.dump(label_encoder, f)
print(f'✅ Label encoder saved     → {ENCODER_FILE}')

# ---- Verify files were created ----
import os
print(f'\n📂 SAVED FILE SIZES:')
for fname in [MODEL_FILE, VECTORIZER_FILE, ENCODER_FILE]:
    size_kb = os.path.getsize(fname) / 1024
    print(f'   {fname:30} : {size_kb:.1f} KB')

# ---- Demonstrate loading ----
print('\n🔄 LOADING SAVED MODEL (demonstration)...')
with open(MODEL_FILE, 'rb') as f:      # 'rb' = read binary mode
    loaded_model = pickle.load(f)

test_acc_loaded = accuracy_score(y_test, loaded_model.predict(X_test_tfidf))
print(f'✅ Loaded model accuracy: {test_acc_loaded*100:.2f}%  (matches original → save/load works!)')

## 💬 Section 14: Response Generation System

Now we build the **brain of our chatbot** — mapping intents to responses!

Each intent has **multiple responses** so the bot doesn't sound repetitive.  
A random response is selected each time.

In [ ]:
# ============================================================
# RESPONSE MAPPING: Intent → Multiple Possible Responses
# ============================================================
# This dictionary maps each intent name to a list of responses.
# The chatbot randomly picks one response each time.

INTENT_RESPONSES = {
    # --- Banking & Account ---
    'transfer': [
        "I can help you transfer money. Please provide the recipient's details.",
        "Sure! Who would you like to transfer money to and how much?",
        "I'll process your transfer. Please verify the account number."
    ],
    'balance': [
        "Your current account balance is $2,450.75.",
        "Let me check your balance. Your available balance is $2,450.75.",
        "Your account balance as of today is $2,450.75."
    ],
    'transactions': [
        "Here are your last 5 transactions: [details would show here].",
        "I can show your transaction history. What date range would you like?",
        "Fetching your recent transactions now..."
    ],
    'freeze_account': [
        "Your account has been frozen. Please contact support to unfreeze it.",
        "Account freeze request received. Your account is now locked for safety.",
        "I've initiated the account freeze. This protects you from unauthorized access."
    ],
    'pin_change': [
        "To change your PIN, go to Settings > Security > Change PIN.",
        "I'll guide you through the PIN change process. Please verify your identity first.",
        "PIN change initiated. Please enter your current PIN to proceed."
    ],
    
    # --- Credit Card ---
    'card_declined': [
        "Your card may have been declined due to insufficient funds or security check.",
        "Sorry to hear that! Common reasons: insufficient balance, expired card, or security flag.",
        "I'll investigate why your card was declined. Please check your balance first."
    ],
    'credit_limit': [
        "Your current credit limit is $5,000. Would you like to request an increase?",
        "I can check your credit limit for you. Currently it's set at $5,000.",
        "Your credit limit information: $5,000 total, $3,200 available."
    ],
    'rewards': [
        "You have 2,450 reward points. Redeem them for cashback or travel miles!",
        "Your rewards balance: 2,450 points worth approximately $24.50.",
        "Great news! You've earned 2,450 reward points on your card."
    ],
    
    # --- Loans & Payments ---
    'pay_bill': [
        "I can help you pay your bill. Which biller and how much?",
        "Bill payment initiated. Please confirm the amount and biller.",
        "Sure! I'll set up the bill payment for you."
    ],
    'loan': [
        "We offer personal loans from $1,000 to $50,000 at competitive rates.",
        "I can provide loan information. What type of loan are you interested in?",
        "For loan applications, I'll need some financial details. Shall we proceed?"
    ],
    'interest_rate': [
        "Current interest rates: Savings 3.5%, Loans from 6.9%, Credit Card 18.9%.",
        "Our current rates are: Savings 3.5% APY, Personal Loan from 6.9% APR.",
        "Interest rate information: please visit our rates page for the most current data."
    ],
    
    # --- Currency & Exchange ---
    'exchange_rate': [
        "Today's exchange rates: USD/EUR 0.92, USD/GBP 0.79, USD/INR 83.5.",
        "Current exchange rates vary. USD to EUR is approximately 0.92 today.",
        "I'll fetch the latest exchange rates for you. Which currencies do you need?"
    ],
    'currency': [
        "I can help with currency conversion. Which currencies would you like?",
        "Currency conversion service available. Please specify the amount and currencies.",
        "Sure! Tell me the amount and I'll convert it to your desired currency."
    ],
    
    # --- Greetings & Small Talk ---
    'greeting': [
        "Hello! How can I assist you today?",
        "Hi there! I'm your virtual banking assistant. What can I help you with?",
        "Good day! Ready to help you with any banking queries.",
        "Hey! Great to see you. How can I be of service?"
    ],
    'goodbye': [
        "Goodbye! Have a wonderful day!",
        "Bye! Feel free to return if you need any assistance!",
        "Take care! Wishing you a great day ahead!"
    ],
    'thank_you': [
        "You're welcome! Is there anything else I can help you with?",
        "Happy to help! Let me know if you need anything else.",
        "My pleasure! Don't hesitate to ask if you need more assistance."
    ],
    'what_can_i_ask_you': [
        "I can help with: balance checks, transfers, bill payments, loan info, and much more!",
        "Ask me about your account, transactions, credit cards, loans, or exchange rates!",
        "I'm your AI banking assistant! I handle 150+ types of banking queries."
    ],
    
    # --- Calendar & Time ---
    'time': [
        f"The current time is {datetime.now().strftime('%H:%M')}.",
        f"It's {datetime.now().strftime('%I:%M %p')} right now.",
        f"Current time: {datetime.now().strftime('%H:%M:%S')}."
    ],
    'date': [
        f"Today's date is {datetime.now().strftime('%B %d, %Y')}.",
        f"It's {datetime.now().strftime('%A, %B %d, %Y')} today.",
        f"The current date is {datetime.now().strftime('%d/%m/%Y')}."
    ],
    'alarm': [
        "I can set an alarm for you! What time should I set it for?",
        "Alarm setting initiated. Please specify the time.",
        "Sure! Tell me the time and I'll set your alarm."
    ],
    
    # --- Utilities ---
    'weather': [
        "I'd recommend checking a weather app for the most accurate forecast!",
        "For weather updates, try weather.com or your phone's weather app.",
        "Weather services show partly cloudy skies. Check your local forecast for details."
    ],
    'calculator': [
        "I can help with calculations! What would you like to compute?",
        "Sure! Give me the numbers and I'll calculate for you.",
        "Calculator ready! What's the calculation?"
    ],
    'definition': [
        "I can help explain terms! What word or concept would you like defined?",
        "Sure! Tell me the term and I'll give you a clear definition.",
        "Definitions available! Which word are you looking to understand?"
    ],
    
    # --- Travel ---
    'flight_status': [
        "I can check flight status! Please provide the flight number.",
        "Flight status check available. What is your flight number?",
        "Tell me your flight number and I'll fetch the status for you."
    ],
    'travel_alert': [
        "Travel alerts are available for your destination. Stay safe!",
        "I can check travel advisories. Which country are you traveling to?",
        "For travel alerts, please specify your destination country."
    ],
    'book_flight': [
        "I can help you book a flight! Where would you like to travel?",
        "Flight booking assistance available. Please provide departure and destination.",
        "Let's book your flight! Where and when are you traveling?"
    ],
    
    # --- Food & Restaurants ---
    'restaurant_suggestion': [
        "I can suggest restaurants near you! What cuisine are you in the mood for?",
        "Great choice! Looking for restaurants. What's your preferred cuisine?",
        "Food time! Tell me your location and preferred cuisine for recommendations."
    ],
    'recipe': [
        "I can help with recipes! What dish would you like to make?",
        "Cooking time! Tell me the dish and I'll find a recipe for you.",
        "Recipe search initiated. What are you planning to cook?"
    ],
    
    # --- Fun & Entertainment ---
    'tell_joke': [
        "Why don't scientists trust atoms? Because they make up everything! 😄",
        "What do you call a fake noodle? An impasta! 🍝😂",
        "Why did the scarecrow win an award? Because he was outstanding in his field! 🌾",
        "What's a computer's favorite snack? Microchips! 💻😄"
    ],
    'meaning_of_life': [
        "The meaning of life is 42! (According to Douglas Adams 😄)",
        "That's a deep question! Philosophers have debated this for centuries.",
        "Life's meaning is subjective — it's what you make of it!"
    ],
    'who_made_you': [
        "I'm an AI chatbot built using Python, NLP, and Machine Learning!",
        "I was created as a B.Tech NLP project using CLINC150 dataset.",
        "A student built me using scikit-learn, TF-IDF, and Logistic Regression!"
    ],
    
    # --- Health ---
    'calories': [
        "For calorie information, please check a nutrition database like MyFitnessPal.",
        "I can help estimate calories! What food would you like to check?",
        "Calorie tracking is important! Tell me the food item for information."
    ],
    'nutrition': [
        "I can provide basic nutrition info. What food are you curious about?",
        "Nutrition facts available! Which food item would you like to check?",
        "Healthy eating starts with knowledge! What would you like to know?"
    ],
    
    # --- General fallback ---
    'oos': [
        "I'm not sure I understood that. Could you rephrase?",
        "That's outside my expertise. Can you ask something else?",
        "Hmm, I don't have a good answer for that. Try asking differently!"
    ]
}

# ---- Default response for unmapped intents ----
DEFAULT_RESPONSES = [
    "I understand your query about {intent}. Let me help you with that!",
    "You're asking about {intent}. I'm looking into that for you.",
    "Regarding {intent}: I'll do my best to assist you!"
]

def get_response(intent_name):
    """
    Get a response for a given intent.
    If the intent is in our mapping → return a random specific response.
    Otherwise → return a generic response with the intent name.
    """
    # Check if we have a specific response for this intent
    for key in INTENT_RESPONSES:
        if key in intent_name.lower() or intent_name.lower() in key:
            return random.choice(INTENT_RESPONSES[key])
    
    # Generic fallback response
    template = random.choice(DEFAULT_RESPONSES)
    return template.format(intent=intent_name.replace('_', ' '))


print('✅ Response generation system created!')
print(f'   Intents with custom responses: {len(INTENT_RESPONSES)}')
print(f'\n🧪 TESTING RESPONSE SYSTEM:')
for intent in ['greeting', 'balance', 'transfer', 'tell_joke', 'card_declined']:
    print(f'   Intent: {intent:20} → Response: {get_response(intent)}')

## 🤖 Section 15: Chatbot Pipeline

Now we connect everything into a single pipeline:

```
User Query → Preprocessing → TF-IDF → Intent Prediction → Confidence Score → Response
```

In [ ]:
# ============================================================
# CHATBOT PIPELINE CLASS
# ============================================================

class IntentChatbot:
    """
    A complete intent-based chatbot.
    
    This class wraps the entire pipeline:
    text → preprocess → tfidf → predict → respond
    """
    
    def __init__(self, model, vectorizer, encoder, confidence_threshold=0.35):
        """
        Initialize the chatbot with trained components.
        
        Parameters:
            model                : Trained classifier (NB or LR)
            vectorizer           : Fitted TF-IDF vectorizer
            encoder              : Fitted LabelEncoder
            confidence_threshold : Min confidence to accept prediction (default: 35%)
        """
        self.model               = model
        self.vectorizer          = vectorizer
        self.encoder             = encoder
        self.threshold           = confidence_threshold
        self.conversation_history = []  # Track conversation
    
    def preprocess(self, text):
        """Clean the user's input text."""
        return preprocess_text(text)
    
    def predict_intent(self, text):
        """
        Predict the intent and confidence for a given text.
        
        Returns:
            intent_name (str)   : The predicted intent label
            confidence  (float) : Probability score (0 to 1)
        """
        # Step 1: Preprocess
        cleaned = self.preprocess(text)
        
        # Step 2: TF-IDF vectorize
        features = self.vectorizer.transform([cleaned])
        
        # Step 3: Predict class
        predicted_class = self.model.predict(features)[0]  # Returns encoded label
        
        # Step 4: Get probability scores for ALL classes
        proba = self.model.predict_proba(features)[0]  # Array of probabilities
        confidence = proba.max()                        # Highest probability = confidence
        
        # Step 5: Decode class number → intent name
        intent_name = self.encoder.inverse_transform([predicted_class])[0]
        
        return intent_name, confidence
    
    def respond(self, user_input, verbose=True):
        """
        Full response pipeline: input → response.
        
        Parameters:
            user_input : The user's message string
            verbose    : If True, print detailed output
        
        Returns:
            response (str), intent (str), confidence (float)
        """
        # Handle empty input
        if not user_input.strip():
            return "Please type something!", "unknown", 0.0
        
        # Predict intent and confidence
        intent, confidence = self.predict_intent(user_input)
        
        # Check confidence threshold (fallback mechanism)
        if confidence < self.threshold:
            response = "⚠️ Sorry, I didn't understand your question. Please rephrase it."
            intent   = "unknown"
        else:
            response = get_response(intent)  # Get response from our response map
        
        # Log the conversation
        self._log_conversation(user_input, intent, confidence, response)
        
        return response, intent, confidence
    
    def _log_conversation(self, user_input, intent, confidence, response):
        """Save conversation to history list."""
        self.conversation_history.append({
            'timestamp'  : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'user_query' : user_input,
            'intent'     : intent,
            'confidence' : round(confidence * 100, 2),
            'response'   : response
        })


# ============================================================
# CREATE CHATBOT INSTANCE
# ============================================================

chatbot = IntentChatbot(
    model       = best_model,
    vectorizer  = tfidf_vectorizer,
    encoder     = label_encoder,
    confidence_threshold = 0.35  # 35% minimum confidence
)

print('🤖 Chatbot initialized successfully!')
print(f'   Model      : {best_model_name}')
print(f'   Threshold  : {chatbot.threshold * 100:.0f}% confidence required')
print(f'   Intents    : {num_classes} intents supported')

# ---- Quick test of the pipeline ----
print('\n🧪 PIPELINE TEST:')
test_query = "What is my current account balance?"
response, intent, confidence = chatbot.respond(test_query)
print(f'   Query      : {test_query}')
print(f'   Intent     : {intent}')
print(f'   Confidence : {confidence*100:.1f}%')
print(f'   Response   : {response}')

## 🛡️ Section 16: Fallback Handling

**Fallback** handles cases where the chatbot is uncertain about the intent.

If confidence < threshold → return a polite "I don't understand" message.  
This prevents the bot from giving **wrong, confident-sounding** answers to unknown queries.

In [ ]:
# ============================================================
# FALLBACK HANDLING DEMONSTRATION
# ============================================================

print('🛡️ FALLBACK HANDLING DEMONSTRATION')
print('=' * 60)
print(f'Confidence threshold: {chatbot.threshold * 100:.0f}%')
print('Queries below this threshold get a fallback response.')
print()

fallback_tests = [
    # These are clearly in-scope queries (should get good confidence)
    "What is my account balance?",
    "Transfer money to my friend",
    "My card was declined",
    # These are vague/random queries (might fall back)
    "asdfghj qwerty random nonsense",
    "xyz abc 123",
    "quantum physics of black holes in outer space",
    "how to make pizza with chocolate sauce"
]

for query in fallback_tests:
    response, intent, confidence = chatbot.respond(query, verbose=False)
    status = '✅ Answered' if intent != 'unknown' else '⚠️ Fallback'
    print(f'  {status} [{confidence*100:5.1f}%] "{query[:45]}"')
    print(f'           Intent: {intent}')
    print(f'           Response: {response[:70]}...' if len(response) > 70 else f'           Response: {response}')
    print()

## 📝 Section 17: Conversation Logging

In [ ]:
# ============================================================
# SAVE CONVERSATION HISTORY TO CSV
# ============================================================

def save_conversation_history(chatbot, filename='conversation_history.csv'):
    """
    Save the chatbot's conversation history to a CSV file.
    
    CSV columns:
        timestamp   : When the conversation happened
        user_query  : What the user typed
        intent      : The predicted intent
        confidence  : Confidence score (0-100%)
        response    : What the bot said
    """
    if not chatbot.conversation_history:
        print('No conversation history to save yet.')
        return
    
    fieldnames = ['timestamp', 'user_query', 'intent', 'confidence', 'response']
    
    with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()         # Write column headers
        writer.writerows(chatbot.conversation_history)  # Write all rows
    
    print(f'✅ Conversation history saved to: {filename}')
    print(f'   Total conversations logged: {len(chatbot.conversation_history)}')


# Save what we've tested so far
save_conversation_history(chatbot)

# ---- Display the conversation log ----
if os.path.exists('conversation_history.csv'):
    history_df = pd.read_csv('conversation_history.csv')
    print('\n📋 CONVERSATION HISTORY LOG:')
    print(history_df[['user_query', 'intent', 'confidence', 'response']].to_string(index=False))

## 📊 Section 18: Visualizations

In [ ]:
# ============================================================
# VISUALIZATION 3: COMBINED DASHBOARD
# ============================================================

fig = plt.figure(figsize=(18, 14))
fig.suptitle('🤖 Intent Classification Chatbot — Results Dashboard', fontsize=16, fontweight='bold', y=0.98)

# ---- Plot 1: Top 20 Intents ----
ax1 = fig.add_subplot(2, 2, 1)
top20 = intent_counts.head(20)
colors_bar = sns.color_palette('husl', 20)
ax1.barh(range(len(top20)), top20.values, color=colors_bar)
ax1.set_yticks(range(len(top20)))
ax1.set_yticklabels([name[:22] for name in top20.index], fontsize=8)
ax1.set_xlabel('Training Samples', fontsize=9)
ax1.set_title('📊 Top 20 Intents by Frequency', fontsize=11, fontweight='bold')
ax1.invert_yaxis()

# ---- Plot 2: Model Comparison Radar ----
ax2 = fig.add_subplot(2, 2, 2)
metrics      = ['Accuracy', 'Precision', 'Recall', 'F1']
nb_vals      = [nb_results['accuracy'], nb_results['precision'], nb_results['recall'], nb_results['f1']]
lr_vals      = [lr_results['accuracy'], lr_results['precision'], lr_results['recall'], lr_results['f1']]
x_pos        = np.arange(len(metrics))
ax2.bar(x_pos - 0.2, [v*100 for v in nb_vals], width=0.4, label='Naive Bayes', color='#3498db', alpha=0.85)
ax2.bar(x_pos + 0.2, [v*100 for v in lr_vals], width=0.4, label='Logistic Regression', color='#2ecc71', alpha=0.85)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(metrics, fontsize=10)
ax2.set_ylabel('Score (%)', fontsize=9)
ax2.set_ylim(0, 110)
ax2.set_title('🏆 Model Performance Comparison', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
for i, (nb_v, lr_v) in enumerate(zip(nb_vals, lr_vals)):
    ax2.text(i - 0.2, nb_v*100 + 1.5, f'{nb_v*100:.1f}', ha='center', fontsize=7, fontweight='bold')
    ax2.text(i + 0.2, lr_v*100 + 1.5, f'{lr_v*100:.1f}', ha='center', fontsize=7, fontweight='bold')

# ---- Plot 3: Training Time ----
ax3 = fig.add_subplot(2, 2, 3)
models_names = ['Naive Bayes', 'Logistic Regression']
times        = [train_time_nb, train_time_lr]
bar_colors   = ['#3498db', '#2ecc71']
bars         = ax3.bar(models_names, times, color=bar_colors, alpha=0.85, edgecolor='white')
for bar, t in zip(bars, times):
    ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
             f'{t:.2f}s', ha='center', fontsize=11, fontweight='bold')
ax3.set_ylabel('Training Time (seconds)', fontsize=9)
ax3.set_title('⏱️ Training Time Comparison', fontsize=11, fontweight='bold')

# ---- Plot 4: Text Length Distribution ----
ax4 = fig.add_subplot(2, 2, 4)
ax4.hist(train_df['word_count'], bins=25, color='#9b59b6', edgecolor='white', alpha=0.85)
ax4.axvline(train_df['word_count'].mean(), color='red', linestyle='--', linewidth=2,
            label=f'Mean = {train_df["word_count"].mean():.1f} words')
ax4.set_xlabel('Words per Query', fontsize=9)
ax4.set_ylabel('Count', fontsize=9)
ax4.set_title('📝 Query Length Distribution', fontsize=11, fontweight='bold')
ax4.legend(fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard saved as dashboard.png')

## 🧪 Section 19: Live Testing Examples

Let's test the chatbot on 20+ custom queries and see how it performs!

In [ ]:
# ============================================================
# LIVE TESTING ON 25 CUSTOM QUERIES
# ============================================================

test_queries = [
    # --- Banking ---
    "What is my current account balance?",
    "Transfer 500 rupees to Rahul's account",
    "Show me my last 10 transactions",
    "My account has been blocked",
    "I want to change my ATM PIN",
    
    # --- Credit Card ---
    "My credit card was declined at the store",
    "What is my credit card limit?",
    "How many reward points do I have?",
    
    # --- Currency & Exchange ---
    "Show me today's exchange rates",
    "Convert 100 dollars to euros",
    
    # --- Loans & Bills ---
    "Pay my electricity bill",
    "What are the loan interest rates?",
    "I want to apply for a personal loan",
    
    # --- Small Talk ---
    "Hello, how are you?",
    "Tell me a joke",
    "What can you help me with?",
    "Thank you for your help",
    "Goodbye",
    
    # --- Date/Time ---
    "What time is it?",
    "What is today's date?",
    
    # --- Food ---
    "How to cook biryani?",
    "Suggest some good restaurants nearby",
    
    # --- Travel ---
    "What is the status of flight AI202?",
    "Book a flight to Mumbai",
    
    # --- Unknown ---
    "Explain quantum entanglement to me"
]

print('🧪 LIVE CHATBOT TESTING — 26 Queries')
print('=' * 75)

results = []
for i, query in enumerate(test_queries, 1):
    response, intent, confidence = chatbot.respond(query, verbose=False)
    results.append({
        'No.': i,
        'Query': query[:45],
        'Intent': intent[:30],
        'Conf%': f'{confidence*100:.1f}%'
    })
    
    print(f'\n  [{i:2d}] 👤 User: {query}')
    print(f'       🤖 Bot :')
    if intent == 'unknown':
        print(f'           ⚠️  {response}')
    else:
        print(f'           Intent     : {intent}')
        print(f'           Confidence : {confidence*100:.1f}%')
        print(f'           Response   : {response}')

print('\n' + '=' * 75)
print('✅ All test queries processed!')

In [ ]:
# ============================================================
# SUMMARY TABLE OF LIVE TEST RESULTS
# ============================================================

results_df = pd.DataFrame(results)
print('📊 LIVE TEST RESULTS SUMMARY:')
print(results_df.to_string(index=False))

# Count answered vs fallback
answered = sum(1 for r in results if r['Conf%'] != '0.0%' and 'unknown' not in r['Intent'])
fallback = len(results) - answered
print(f'\n📊 SUMMARY:')
print(f'   Total queries  : {len(results)}')
print(f'   Answered       : {answered} ({answered/len(results)*100:.0f}%)')
print(f'   Fallback       : {fallback} ({fallback/len(results)*100:.0f}%)')

# Save updated conversation history
save_conversation_history(chatbot)

## 📝 Section 20: Results and Conclusion

In [ ]:
# ============================================================
# FINAL RESULTS SUMMARY
# ============================================================

print('=' * 65)
print('📊  FINAL PROJECT RESULTS & CONCLUSION')
print('=' * 65)

print(f"""
🏆 BEST MODEL: {best_model_name}

📊 TEST SET PERFORMANCE:
   Accuracy  : {lr_results['accuracy']*100:.2f}%
   Precision : {lr_results['precision']*100:.2f}%
   Recall    : {lr_results['recall']*100:.2f}%
   F1 Score  : {lr_results['f1']*100:.2f}%

📂 DATASET:
   Name      : CLINC150 (plus variant)
   Training  : {len(train_df):,} samples
   Testing   : {len(test_df):,} samples
   Intents   : {num_classes} unique intent classes

✅ STRENGTHS:
   1. Fast training and inference (milliseconds per query)
   2. High accuracy on the diverse CLINC150 benchmark
   3. Confidence scores help identify uncertain predictions
   4. Fallback handling prevents wrong answers
   5. Clean, modular code — easy to extend
   6. No deep learning required — runs on any laptop

⚠️  LIMITATIONS:
   1. Doesn't understand context across multiple messages
   2. Struggles with typos and misspellings
   3. TF-IDF misses word order and semantics
   4. Fixed response templates (not generative)
   5. Performance drops on very short queries
   6. No multi-language support
""")

print('=' * 65)

## 🚀 Section 21: Future Improvements

This project is a solid foundation. Here's how to take it further:

In [ ]:
print("""🚀 FUTURE IMPROVEMENTS ROADMAP
================================================

1. 🧠 BERT & TRANSFORMERS (Immediate Next Step)
   ─────────────────────────────────────────────
   What  : Use pre-trained language models (BERT, RoBERTa, DistilBERT)
   Why   : BERT understands context and word order — much smarter than TF-IDF
   How   : pip install transformers; from transformers import BertForSequenceClassification
   Gain  : +5-15% accuracy improvement expected
   Code  : 
     from transformers import pipeline
     classifier = pipeline('text-classification', model='bert-base-uncased')

2. 🌐 STREAMLIT WEB APP DEPLOYMENT
   ─────────────────────────────────────────────
   What  : Turn this notebook into a beautiful web chatbot UI
   Why   : Share your project with anyone — no Jupyter needed!
   How   : pip install streamlit
   Code  :
     import streamlit as st
     user_input = st.text_input('You: ')
     if user_input:
         response, intent, conf = chatbot.respond(user_input)
         st.write(f'Bot: {response}')
   Run   : streamlit run app.py

3. 🔌 FLASK REST API
   ─────────────────────────────────────────────
   What  : Create an API endpoint for the chatbot
   Why   : Integrate chatbot into any website or mobile app
   How   : pip install flask
   Code  :
     from flask import Flask, request, jsonify
     app = Flask(__name__)
     @app.route('/chat', methods=['POST'])
     def chat():
         msg = request.json['message']
         response, intent, conf = chatbot.respond(msg)
         return jsonify({'response': response, 'intent': intent, 'confidence': conf})

4. 🔗 DEEP LEARNING APPROACHES
   ─────────────────────────────────────────────
   a) BiLSTM: Bidirectional LSTM captures word order both ways
   b) CNN for text: Captures local patterns (n-grams) efficiently
   c) Sentence-BERT: Semantic embeddings for better similarity
   d) GPT-based: Generative responses (not just templates!)

5. 📊 DATA IMPROVEMENTS
   ─────────────────────────────────────────────
   • Add spell correction (pip install pyspellchecker)
   • Data augmentation to increase training examples
   • Multi-language support (translate queries first)
   • Entity extraction (recognize names, amounts, dates)

6. 🔧 ENGINEERING IMPROVEMENTS
   ─────────────────────────────────────────────
   • Docker containerization for easy deployment
   • Database integration for persistent conversation memory
   • User authentication and personalized responses
   • A/B testing different models in production

================================================
Start with Streamlit deployment — it gives the most
impressive demo for your resume in just 1-2 hours!
================================================
""")

---
## 💬 Interactive Chat Window
### Live Chatbot — Type your messages below!

> **Instructions:**
> - Run the cell below
> - Type your message and press Enter
> - Type `exit`, `quit`, or `bye` to stop

---

In [ ]:
# ============================================================
# 🤖 INTERACTIVE CHAT WINDOW
# ============================================================
# This creates a live chat session directly inside Jupyter!
# Type your messages and the bot will respond.
# Commands to exit: 'exit', 'quit', 'bye'
# ============================================================

EXIT_COMMANDS = {'exit', 'quit', 'bye', 'goodbye', 'stop', 'end'}

def format_bot_response(response, intent, confidence):
    """
    Format the bot's reply for clean display in the notebook.
    """
    output = []
    output.append('─' * 60)
    output.append('🤖 Bot:')
    if intent == 'unknown':
        output.append(f'   {response}')
    else:
        output.append(f'   Intent     : {intent}')
        output.append(f'   Confidence : {confidence*100:.1f}%')
        output.append(f'   Response   : {response}')
    output.append('─' * 60)
    return '\n'.join(output)


# ---- WELCOME MESSAGE ----
print('=' * 60)
print('  🤖 INTENT CLASSIFICATION CHATBOT — LIVE CHAT')
print('  Powered by:', best_model_name)
print('  Intents   :', num_classes, 'supported')
print('  Type exit / quit / bye to stop')
print('=' * 60)
print()

# ---- MAIN CHAT LOOP ----
while True:
    try:
        # Get user input
        user_input = input('👤 You: ').strip()
        
        # Skip empty input
        if not user_input:
            print('   (Please type something)\n')
            continue
        
        # Check for exit commands
        if user_input.lower() in EXIT_COMMANDS:
            print('\n🤖 Bot: Thank you for chatting with me! Goodbye! 👋')
            print('\n✅ Chat session ended.')
            
            # Save final conversation history
            save_conversation_history(chatbot)
            print(f'📝 {len(chatbot.conversation_history)} messages saved to conversation_history.csv')
            break
        
        # Get chatbot response
        response, intent, confidence = chatbot.respond(user_input)
        
        # Display formatted response
        print(format_bot_response(response, intent, confidence))
        print()  # Empty line for readability
        
    except KeyboardInterrupt:
        # Handle Ctrl+C gracefully
        print('\n\n🤖 Bot: Chat interrupted. Goodbye! 👋')
        save_conversation_history(chatbot)
        print(f'📝 Conversation saved to conversation_history.csv')
        break
    except EOFError:
        # Handle non-interactive environments (like automated testing)
        print('\n[Non-interactive mode detected — skipping live chat]')
        print('To use the chatbot, run this cell in a Jupyter Notebook!')
        break

In [ ]:
# ============================================================
# FINAL SUMMARY — ALL FILES CREATED
# ============================================================

print('\n🎉 PROJECT COMPLETE!')
print('=' * 60)
print('\n📂 FILES GENERATED:')

files_to_check = [
    ('best_model.pkl',            'Trained classifier'),
    ('tfidf_vectorizer.pkl',      'TF-IDF vectorizer'),
    ('label_encoder.pkl',         'Label encoder'),
    ('conversation_history.csv',  'Chat history log'),
    ('intent_distribution.png',   'Intent distribution plot'),
    ('confusion_matrix.png',      'Confusion matrix'),
    ('model_comparison.png',      'Model comparison chart'),
    ('dashboard.png',             'Full results dashboard'),
]

for fname, description in files_to_check:
    if os.path.exists(fname):
        size = os.path.getsize(fname)
        print(f'  ✅ {fname:35} [{size/1024:6.1f} KB] — {description}')
    else:
        print(f'  ❌ {fname:35} — NOT FOUND')

print(f'\n🏆 Best Model    : {best_model_name}')
print(f'📊 Test Accuracy : {lr_results["accuracy"]*100:.2f}%')
print(f'📊 Test F1 Score : {lr_results["f1"]*100:.2f}%')
print(f'💬 Chat Sessions : {len(chatbot.conversation_history)} messages')
print('\n✨ Ready to add to your resume! ✨')
print('=' * 60)